In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
print("Loading datasets...")

race_df = pd.read_csv('data/processed/race_results_2025-6_clean.csv')
reddit_df = pd.read_csv('data/raw/reddit_posts.csv')
youtube_df = pd.read_csv('data/raw/youtube_videos_sponsor_2025-6.csv')
news_df = pd.read_csv('data/raw/news_mentions_2025-6_raw.csv')

print(f"Race Results: {len(race_df)} rows")
print(f"Reddit Mentions: {len(reddit_df)} rows")
print(f"YouTube Engagement: {len(youtube_df)} rows")
print(f"News Mentions: {len(news_df)} rows")

In [ ]:
def audit_dataset(df, name):
    """
    Print comprehensive audit of dataset structure.
    """
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nColumn Names and Types:")
    print("-" * 40)
    for col in df.columns:
        dtype = df[col].dtype
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "N/A"
        print(f"  {col:25} | {str(dtype):10} | Sample: {str(sample)[:30]}")
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("  None")
    return None

# Audit each dataset
audit_dataset(race_df, "Race Results")
audit_dataset(reddit_df, "Reddit Mentions")
audit_dataset(youtube_df, "YouTube Engagement")
audit_dataset(news_df, "News Mentions")

In [ ]:
# Define standard date format
DATE_FORMAT = '%Y-%m-%d'  # ISO 8601: 2024-02-18

def standardize_dates(df, date_columns, parse_format=None):
    """
    Convert all date columns to standard format.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    date_columns : list
        List of column names containing dates
    parse_format : str, optional
        Explicit format to use when parsing the input column (e.g. '%Y%m%d%H%M%S').
        Needed for numeric-looking date columns like 'seendate', since pandas
        otherwise treats an int64 column as nanoseconds since epoch instead of
        parsing it as a date string.

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized dates
    """
    df = df.copy()

    for col in date_columns:
        if col not in df.columns:
            print(f"  Warning: Column '{col}' not found")
            continue

        # Convert to datetime
        if parse_format:
            df[col] = pd.to_datetime(df[col].astype(str), format=parse_format, errors='coerce')
        else:
            df[col] = pd.to_datetime(df[col], errors='coerce')

        # Check for conversion failures
        failed = df[col].isna().sum()
        if failed > 0:
            print(f"  Warning: {failed} dates failed to convert in '{col}'")

        # Convert to standard string format for CSV compatibility
        df[f'{col}_str'] = df[col].dt.strftime(DATE_FORMAT)

    return df

# Apply to each dataset
print("Standardizing dates...")

race_df = standardize_dates(race_df, ['Race_Date'])
reddit_df = standardize_dates(reddit_df, ['race_date'] if 'race_date' in reddit_df.columns else [])
youtube_df = standardize_dates(youtube_df, ['race_date'])

news_df = standardize_dates(news_df, ['seendate'], parse_format='%Y%m%d%H%M%S')

print("Date standardization complete.")

In [ ]:
def standardize_columns(df, column_mapping):
    """
    Rename columns to standard names and convert to lowercase with underscores.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    column_mapping : dict
        Dictionary mapping old names to new names

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized column names
    """
    df = df.copy()

    # Apply explicit mappings
    df = df.rename(columns=column_mapping)

    # Convert remaining columns to lowercase with underscores
    df.columns = df.columns.str.lower().str.replace(' ', '_')

    return df

# Define standard column names for each dataset
race_columns = {
    'Race_Name': 'race_name',
    'Race_Date': 'race_date',
    'Race_Number': 'race_number',
    'Driver': 'driver',
    'Team': 'team',
    'Sponsor': 'sponsor',
    'Finish_Position': 'finish_position',
    'Laps_Led': 'laps_led'
}

exposure_columns = {
    'race_period': 'race_name',  # If Reddit uses race_period
    'Race_Name': 'race_name',
    'Race_Number': 'race_number',
    'Sponsor': 'sponsor'
}

# Apply standardization
race_df = standardize_columns(race_df, race_columns)
reddit_df = standardize_columns(reddit_df, exposure_columns)
youtube_df = standardize_columns(youtube_df, exposure_columns)
news_df = standardize_columns(news_df, exposure_columns)

print("Column name standardization complete.")

In [ ]:
# add race numbers based on date:
date_to_number = {pd.Timestamp('2025-02-1'): 1,
                  pd.Timestamp('2025-2-12'): 2,
                  pd.Timestamp('2025-2-22'): 3,
                  pd.Timestamp('2025-2-28'): 4,
                  pd.Timestamp('2025-3-8'): 5,
                  pd.Timestamp('2025-3-15'): 6,
                  pd.Timestamp('2025-3-22'): 7,
                  pd.Timestamp('2025-3-30'): 8,
                  pd.Timestamp('2025-4-5'): 9,
                  pd.Timestamp('2025-4-12'): 10,
                  pd.Timestamp('2025-4-26'): 11,
                  pd.Timestamp('2025-5-3'): 12,
                  pd.Timestamp('2025-5-10'): 13,
                  pd.Timestamp('2025-5-17'): 14,
                  pd.Timestamp('2025-5-24'): 15,
                  pd.Timestamp('2025-5-31'): 16,
                  pd.Timestamp('2025-6-7'): 17,
                  pd.Timestamp('2025-6-14'): 18,
                  pd.Timestamp('2025-6-21'): 19,
                  pd.Timestamp('2025-6-27'): 20,
                  pd.Timestamp('2025-7-5'): 21,
                  pd.Timestamp('2025-7-12'): 22,
                  pd.Timestamp('2025-7-19'): 23,
                  pd.Timestamp('2025-7-26'): 24,
                  pd.Timestamp('2025-8-2'): 25,
                  pd.Timestamp('2025-8-9'): 26,
                  pd.Timestamp('2025-8-15'): 27,
                  pd.Timestamp('2025-8-22'): 28,
                  pd.Timestamp('2025-8-30'): 29,
                  pd.Timestamp('2025-9-6'): 30,
                  pd.Timestamp('2025-9-12'): 31,
                  pd.Timestamp('2025-9-20'): 32,
                  pd.Timestamp('2025-9-27'): 33,
                  pd.Timestamp('2025-10-4'): 34,
                  pd.Timestamp('2025-10-12'): 35,
                  pd.Timestamp('2025-10-18'): 36,
                  pd.Timestamp('2025-10-27'): 37,
                  pd.Timestamp('2025-11-1'): 38,
                  pd.Timestamp('2026-2-21'): 39,
                  pd.Timestamp('2026-2-28'): 40,
                  pd.Timestamp('2026-3-7'): 41,
                  pd.Timestamp('2026-3-14'): 42,
                  pd.Timestamp('2026-3-21'): 43,
                  pd.Timestamp('2026-3-28'): 44,
                  pd.Timestamp('2026-4-11'): 45,
                  pd.Timestamp('2026-4-18'): 46,
                  pd.Timestamp('2026-4-25'): 47,
                  pd.Timestamp('2026-5-2'): 48,
                  pd.Timestamp('2026-5-9'): 49,
                  pd.Timestamp('2026-5-16'): 50,
                  pd.Timestamp('2026-5-23'): 51,
                  pd.Timestamp('2026-5-30'): 52,
                  pd.Timestamp('2026-6-6'): 53,
}
race_series = pd.Series(date_to_number).sort_index()

def assign_race_number(df, date_col='published')-> pd.DataFrame:
    """Assign race numbers based on the most recent race date on or before the given date column."""
    df = df.copy()
    normalized = pd.to_datetime(df[date_col], utc=True).dt.tz_localize(None).dt.normalize()
    df['race_number'] = normalized.apply(lambda d: race_series.asof(d))
    return df
reddit_df = assign_race_number(reddit_df, 'published')
news_df = assign_race_number(news_df, 'seendate')

In [ ]:
race_num_to_name = (race_df[['race_number', 'race_name']].drop_duplicates().set_index('race_number')['race_name']).to_dict()

def assign_race_name(df, race_num_col='race_number')-> pd.DataFrame:
    """Assign race names based on race numbers."""
    df = df.copy()
    df['race_name'] = df[race_num_col].map(race_num_to_name)
    return df

reddit_df = assign_race_name(reddit_df, 'race_number')
news_df = assign_race_name(news_df, 'race_number')
youtube_df = assign_race_name(youtube_df, 'race_number')

In [ ]:
def create_race_name_mapping(df, race_col='race_name'):
    """
    Create a standardized mapping for race names.
    Returns dict mapping original names to standardized names.
    """
    unique_races = df[race_col].unique()
    print(f"Found {len(unique_races)} unique race names:")
    for race in sorted(unique_races):
        print(f"  - {race}")
    return unique_races

# Check race names in each dataset
print("\n=== Race Names in Race Results ===")
race_names_main = create_race_name_mapping(race_df)

print("\n=== Race Names in Reddit Data ===")
race_names_reddit = create_race_name_mapping(reddit_df)

print("\n=== Race Names in YouTube Data ===")
race_names_youtube = create_race_name_mapping(youtube_df)

print("\n=== Race Names in News Data ===")
race_names_news = create_race_name_mapping(news_df)

In [ ]:
# Create master race name mapping
# This ensures all datasets use exactly the same race names

RACE_NAME_MAPPING = {
    # Add variations you find in your data
    '2025 Daytona 500': '2025 Daytona 500',
    '2025 DAYTONA 500': '2025 Daytona 500',
    '2025 Ambetter Health 400': '2025 Ambetter Health 400',
    '2025 Ambetter 400': '2025 Ambetter Health 400',
    '2025 Pennzoil 400': '2025 Pennzoil 400',
    '2025 Pennzoil 400 Las Vegas': '2025 Pennzoil 400',
    '2025 Wurth 400 presented by LIQUI MOLLY': '2025 Wurth 400',
    '2025 Quaker State 400 available at Walmart': '2025 Quaker State 400',
    '2025 Toyota / Save Mart 350': '2025 Toyota Save Mart 350',
    '2025 Brickyard 400 Presented by PPG': '2025 Brickyard 400',
    '2026 Autotrader 400 at EchoPark Speedway': '2026 Autotrader 400',
    '2026 DuraMax Texas Grand Prix Presented by RelaDyne': '2026 DuraMax Texas Grand Prix',
    '2026 Pennzoil 400 Presented by Jiffy Lube': '2026 Pennzoil 400',
    '2026 Wurth 400 Presented by LIQUI MOLY': '2026 Wurth 400',
    
}

def standardize_race_names(df, mapping, race_col='race_name'):
    """
    Apply race name standardization.
    """
    df = df.copy()

    # Apply mapping where available
    df[race_col] = df[race_col].replace(mapping)

    # Check for unmapped races
    unmapped = df[~df[race_col].isin(mapping.values())][race_col].unique()
    if len(unmapped) > 0:
        print(f"Warning: {len(unmapped)} race names not in mapping:")
        for name in unmapped:
            print(f"  - '{name}'")

    return df

# Apply to all datasets
race_df = standardize_race_names(race_df, RACE_NAME_MAPPING)
reddit_df = standardize_race_names(reddit_df, RACE_NAME_MAPPING)
youtube_df = standardize_race_names(youtube_df, RACE_NAME_MAPPING)
news_df = standardize_race_names(news_df, RACE_NAME_MAPPING)

In [33]:
# Sponsor name standardization
SPONSOR_NAME_MAPPING = {
    'FedEx': 'FedEx',
    'Fedex': 'FedEx',
    'FEDEX': 'FedEx',
    'NAPA': 'NAPA',
    'NAPA Auto Parts': 'NAPA',
    'Napa': 'NAPA',
    "McDonald's": "McDonald's",
    'McDonalds': "McDonald's",
    "Mcdonald's": "McDonald's",
    'Ally': 'Ally',
    'Ally Financial': 'Ally',
    'Ally Racing': 'Ally',
    'Busch Light': 'Busch Light',
    'Busch': 'Busch Light',
    'Busch Light Apple': 'Busch Light',
    'progressive': 'Progressive',
    'Progressive Insurance': 'Progressive',
    'progressive insurance': 'Progressive',
    'Progressive insurance': 'Progressive',
    'castrol': 'Castrol',
    'Love\'s Travel Stops': 'Love\'s Travel Stops',
    'Loves Travel Stops': 'Love\'s Travel Stops',
    'love\'s travel stops': 'Love\'s Travel Stops',
    'Love\'s': 'Love\'s Travel Stops',
    'Love’s Travel Stops': 'Love\'s Travel Stops',
    'love’s travel stops': 'Love\'s Travel Stops',
    'Love’s': 'Love\'s Travel Stops',
    'love’s': 'Love\'s Travel Stops',
    'Cheddar\'s Scratch Kitchen': 'Cheddar\'s Scratch Kitchen',
    'Cheddar’s Scratch Kitchen': 'Cheddar\'s Scratch Kitchen',
    'Cheddar’s': 'Cheddar\'s Scratch Kitchen',
    'Cheddar\'s': 'Cheddar\'s Scratch Kitchen',
    'cheddar’s': 'Cheddar\'s Scratch Kitchen',
    'cheddar\'s': 'Cheddar\'s Scratch Kitchen',
    'cheddar’s scratch kitchen': 'Cheddar\'s Scratch Kitchen',
    'cheddar\'s scratch kitchen': 'Cheddar\'s Scratch Kitchen',
}

def standardize_sponsor_names(df, mapping, sponsor_col='sponsor'):
    """
    Apply sponsor name standardization.
    """
    df = df.copy()
    df[sponsor_col] = df[sponsor_col].replace(mapping)

    # Verify all sponsors are standardized
    target_sponsors = ['Progressive', 'Castrol', 'Cheddar\'s Scratch Kitchen', 'Love\'s Travel Stops', 'Busch Light']
    unique_sponsors = df[sponsor_col].unique()
    unexpected = [s for s in unique_sponsors if s not in target_sponsors]
    if unexpected:
        print(f"Warning: Unexpected sponsors found: {unexpected}")

    return df

# Apply to all datasets
reddit_df = standardize_sponsor_names(reddit_df, SPONSOR_NAME_MAPPING)
news_df = standardize_sponsor_names(news_df, SPONSOR_NAME_MAPPING)


In [36]:
def consolidate_sponsor_mentions(df, sponsor_names, prefix='features_', new_col='sponsor'):
    """
    Collapse the per-sponsor boolean mention columns (e.g. 'features_progressive',
    'features_castrol') into a single column holding the sponsor name(s) mentioned
    in that row.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing one boolean column per sponsor
    sponsor_names : list
        Canonical sponsor names (e.g. from SPONSOR_NAME_MAPPING), used to recover
        proper casing/punctuation lost when column names were lowercased
    prefix : str
        Prefix shared by all the mention columns
    new_col : str
        Name of the resulting sponsor column

    Returns:
    --------
    pd.DataFrame : DataFrame with the mention columns replaced by `new_col`
    """
    df = df.copy()

    mention_cols = [c for c in df.columns if c.startswith(prefix)]
    if not mention_cols:
        print(f"Warning: no columns found with prefix '{prefix}'")
        return df

    # Map each column's normalized suffix back to its canonical sponsor name
    normalized_to_canonical = {name.lower().replace(' ', '_'): name for name in sponsor_names}
    col_to_sponsor = {col: normalized_to_canonical.get(col[len(prefix):], col[len(prefix):]) for col in mention_cols}

    def sponsors_in_row(row):
        mentioned = [col_to_sponsor[c] for c in mention_cols if row[c]]
        return ', '.join(mentioned) if mentioned else None

    df[new_col] = df[mention_cols].apply(sponsors_in_row, axis=1)
    df = df.drop(columns=mention_cols)

    return df

# Collapse features_* columns into a single 'sponsor' column
youtube_df = consolidate_sponsor_mentions(youtube_df, sponsor_names=list(SPONSOR_NAME_MAPPING.values()))
print("Sponsors found in YouTube data:")
print(youtube_df['sponsor'].value_counts(dropna=False))

Sponsors found in YouTube data:
sponsor
NaN                                 942
Cheddar's Scratch Kitchen           259
Progressive                         244
Love's Travel Stops                 229
Busch Light                         197
Castrol                              70
Progressive, Busch Light              2
Progressive, Love's Travel Stops      1
Name: count, dtype: int64


In [38]:
def create_merge_key(df, components=['race_number', 'sponsor']):
    """
    Create a unique merge key from multiple columns.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to add merge key to
    components : list
        Columns to combine into merge key

    Returns:
    --------
    pd.DataFrame : DataFrame with merge_key column added
    """
    df = df.copy()

    # Ensure components exist
    missing = [c for c in components if c not in df.columns]
    if missing:
        print(f"Warning: Columns missing for merge key: {missing}")
        return df

    # Create merge key
    df['merge_key'] = df[components].fillna('unknown').astype(str).agg('_'.join, axis=1)

    return df

# Add merge keys to all datasets
race_agg = race_df.groupby(['race_number', 'sponsor']).first().reset_index()
race_agg = create_merge_key(race_agg)

reddit_df = create_merge_key(reddit_df)
youtube_df = create_merge_key(youtube_df)
news_df = create_merge_key(news_df)

print("Sample merge keys:")
print(race_agg['merge_key'].head(15).tolist())

Sample merge keys:
['1_AdventHealth', '1_Ally', '1_Axalta', '1_Bass Pro Shops', '1_Bass Pro Shops/Tracker Boats', '1_Bass Pro Shops/Winchester', '1_BuildSubmarines.com', '1_Busch Light', '1_Carvana', '1_Celsius', "1_Chili's Ride the 'Dente", '1_DEWALT - Interstate Batteries', '1_Discount Tire', '1_Dollar Tree', '1_Fastenal']


In [39]:
# Save standardized versions
race_df.to_csv('data/processed/race_results_standardized.csv', index=False)
reddit_df.to_csv('data/processed/reddit_mentions_standardized.csv', index=False)
youtube_df.to_csv('data/processed/youtube_engagement_standardized.csv', index=False)
news_df.to_csv('data/processed/news_mentions_standardized.csv', index=False)

print("Standardized datasets saved.")

# Document the standardization rules
standardization_log = {
    'date_format': DATE_FORMAT,
    'race_name_mapping_count': len(RACE_NAME_MAPPING),
    'sponsor_name_mapping_count': len(SPONSOR_NAME_MAPPING),
    'merge_key_components': ['race_number', 'sponsor'],
    'standardization_date': datetime.now().isoformat()
}

import json
with open('data/processed/standardization_log.json', 'w') as f:
    json.dump(standardization_log, f, indent=2)

print("Standardization log saved.")

Standardized datasets saved.
Standardization log saved.


In [41]:
# Load standardized data
race_df = pd.read_csv('data/processed/race_results_standardized.csv')
reddit_df = pd.read_csv('data/processed/reddit_mentions_standardized.csv')
youtube_df = pd.read_csv('data/processed/youtube_engagement_standardized.csv')
news_df = pd.read_csv('data/processed/news_mentions_standardized.csv')

# Create race-level summary from race results
# (Each row = one sponsor at one race)
race_base = race_df.groupby(['race_number', 'race_name', 'race_date', 'sponsor', 'driver']).agg({
    'finish_position': 'first',  # Should be one per sponsor per race
    'laps_led': 'sum'  # Total laps led at that race
}).reset_index()

# Add merge key
race_base['merge_key'] = race_base['race_number'].astype(str) + '_' + race_base['sponsor']

print(f"Base dataset: {len(race_base)} sponsor-race combinations")
print(f"Expected: {36} races x {5} sponsors = {36*5} rows")
print(f"Actual: {len(race_base)} rows")

# Check for any sponsor-race combinations
print("\nSponsor coverage:")
print(race_base.groupby('sponsor').size())

Base dataset: 2013 sponsor-race combinations
Expected: 36 races x 5 sponsors = 180 rows
Actual: 2013 rows

Sponsor coverage:
sponsor
1-800-PACK-RAT                1
3D Systems                    1
7-Eleven                      1
A&W Root Beer                 2
AAA Insurance                 1
                             ..
zone / GetGo                  1
zone Jalapeño Lime            4
zone Watermelon x Circle K    1
zone nicotine pouches         1
–                             6
Length: 526, dtype: int64
